# FactChecker в Google Colab
Запускайте ячейки сверху вниз. Проект скачивается из GitHub.
Во второй ячейке введите ключи OpenRouter и Tavily. Модель — DeepSeek V4.1 Flash.


In [ ]:
# 1. Скачать проект из GitHub и установить зависимости.
import subprocess
import sys
from pathlib import Path

project = Path("/content/factchecker-openrouter")
repo = "https://github.com/Viacheslav-hub-blip/factchecker.git"
if (project / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=project, check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", repo, str(project)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    cwd=project,
    check=True,
)


In [ ]:
# 2. Ввести ключи.
import os
from getpass import getpass

for name, prompt in (
    ("OPENROUTER_API_KEY", "Ключ OpenRouter: "),
    ("TAVILY_API_KEY", "Ключ Tavily: "),
):
    value = getpass(prompt).strip()
    if not value:
        raise ValueError(f"Введите значение {name}.")
    os.environ[name] = value
del value
os.environ["OPENROUTER_MODEL"] = "deepseek/deepseek-v4.1-flash-1017"


In [ ]:
# 3. Запустить API и дождаться готовности.
import time
import httpx

if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)

base_url = "http://127.0.0.1:8000"
log_path = Path("/tmp/factchecker.log")
with log_path.open("w", encoding="utf-8") as log:
    server = subprocess.Popen([sys.executable, "main.py"], cwd=project, stdout=log, stderr=subprocess.STDOUT)

ready = False
for _ in range(30):
    if server.poll() is not None:
        break
    try:
        if httpx.get(f"{base_url}/openapi.json", timeout=1).is_success:
            ready = True
            print("Сервер готов. Выполните следующую ячейку.")
            break
    except httpx.RequestError:
        pass
    time.sleep(1)

if not ready:
    print(log_path.read_text(encoding="utf-8", errors="replace")[-3000:])
    raise RuntimeError("Проверьте журнал запуска.")


### Диалог
Запускайте следующую ячейку для каждого сообщения. Начните с «Что ты умеешь?».
Номер беседы подставляется автоматически. Чтобы начать новую, выполните `thread_id = None` в отдельной ячейке.
История в SQLite хранится в среде Colab и исчезнет при удалении этой среды.


In [ ]:
message = input("Ваш вопрос: ").strip()
if message:
    response = httpx.post(
        f"{base_url}/chat",
        json={"message": message, "thread_id": globals().get("thread_id")},
        timeout=360,
    )
    if response.is_success:
        result = response.json()
        thread_id = result["thread_id"]
        print(result["answer"])
    else:
        print(f"HTTP {response.status_code}: {response.text}")
        print(log_path.read_text(encoding="utf-8", errors="replace")[-2000:])
